# FleetMind-LLM｜Phase 6 大纲

**目标：把已完成的实验整理成可核查、可展示的 GitHub 项目，并提炼能用于大模型算法实习简历的成果。** 按顺序推进，每步完成后再进入下一步。

| 步骤 | 核心工作 | 产出 |
|---|---|---|
| **6.1 证据盘点** | 找齐数据、E1/E2/E3 配置、训练记录、指标、逐题预测和哈希；核对四方法对比的评分口径 | 结果与证据对应清单 |
| **6.2 补充分析** | 统计三个训练集的模板占比、E1 Test 中两种时间顺序模板的表现；核对训练 loss 的计算范围 | 简短分析文档，区分观察与推测 |
| **6.3 仓库整理** | 整理 Notebook、配置、数据说明、结果和报告；检查拟公开内容是否含真实业务或个人信息 | 精简、清楚的仓库目录 |
| **6.4 README 展示** | 首屏展示四方法对比，再说明 1,600／4,000／8,000 训练规模实验、E1 选型、最终 Test 与局限 | 一眼能看懂成果的项目首页 |
| **6.5 复核与发布** | 从保存的预测复算指标，检查文档路径和复现步骤，定稿简历要点 | GitHub 发布版与简历表述 |

贯穿整个阶段的原则是：**Val 用于选型，Test 只报告已选定 E1 的结果；合成同源 Test 与人工困难样本分开；原 Phase 5 报告保留原样，新增分析另写。** E3 错误集中在 `validation_B_2` 是观察结果，原因须以进一步统计为依据。

我们现在处于 **6.1**。完成标准很简单：README 中每个准备公开的关键数字，都能指向对应的指标文件和逐题预测。

## 1. 盘点与核对实验文件

**作用**：确认六个实验结果包及其内部证据完整。

**方法**：检查 ZIP 完整性、文件清单，并核对 E1 adapter、固定 Val 和 Test 数据的 SHA256。

**目的**：建立 README 结果与原始文件之间的对应关系。

本步骤只读取已有文件，不训练或生成新预测。报告数字须在后续步骤结合逐题预测复算。

In [ ]:
from pathlib import Path
import hashlib
import json
import re
import zipfile

# 改为六个 ZIP 所在的文件夹；直接上传 Colab 时通常是 /content
ROOT = Path("/content")

# 用原始名称识别文件，自动兼容文件名末尾的 (1)、(2)、(3)
required = {
    "FleetMind_LLM_Phase3_Results.zip": "E1 adapter 与训练记录",
    "FleetMind_LLM_Phase5_E2_Results.zip": "E2 adapter 与训练记录",
    "FleetMind_LLM_Phase5_E3_Results.zip": "E3 adapter 与训练记录",
    "FleetMind_LLM_Phase5_Training_Data.zip": "训练数据与固定 Val",
    "FleetMind_LLM_Phase4_Evaluation_Dataset.zip": "封存 Test 与数据划分",
    "FleetMind_LLM_Phase5_Evaluation_Results.zip": "选型、指标与逐题预测",
}

def original_name(filename):
    """去掉 ZIP 文件名末尾由重复下载产生的编号。"""
    return re.sub(r"\(\d+\)(?=\.zip$)", "", filename)

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

# 只检查 ROOT 下的 ZIP，避免扫描整个 Drive
candidates = {}
for path in ROOT.glob("*.zip"):
    candidates.setdefault(original_name(path.name), []).append(path)

found = {}
print("=== 六个结果包 ===")

for expected, purpose in required.items():
    matches = candidates.get(expected, [])
    print(f"\n{purpose}：")

    if not matches:
        print("  未找到；请检查 ROOT 或先将文件上传到 Colab")
        continue

    if len(matches) > 1:
        print("  发现多个同名版本，请先确定要使用哪一个：")
        for path in matches:
            print("   ", path.name)
        continue

    path = matches[0]
    found[expected] = path

    try:
        with zipfile.ZipFile(path) as archive:
            bad_file = archive.testzip()
            members = [n for n in archive.namelist()
                       if not n.endswith("/")]

        print(f"  实际文件名：{path.name}")
        print(f"  ZIP 完整性：{'通过' if bad_file is None else '失败：' + bad_file}")
        print(f"  内部文件数：{len(members)}")
        for name in members:
            print("   ", name)

    except zipfile.BadZipFile:
        print("  ZIP 完整性：失败，无法读取")
        found.pop(expected)

print(f"\n已识别且可读取：{len(found)}/6 个结果包")

# 以下检查三个跨文件的关键证据。
# 找不到文件时只提示，不把“未检查”误写成“通过”。
print("\n=== 关键哈希交叉核对 ===")

e1_path = found.get("FleetMind_LLM_Phase3_Results.zip")
eval_path = found.get("FleetMind_LLM_Phase5_Evaluation_Results.zip")
data_path = found.get("FleetMind_LLM_Phase5_Training_Data.zip")
test_path = found.get("FleetMind_LLM_Phase4_Evaluation_Dataset.zip")

if e1_path and eval_path:
    with zipfile.ZipFile(e1_path) as e1_zip, \
         zipfile.ZipFile(eval_path) as eval_zip:

        selection_name = next(
            n for n in eval_zip.namelist()
            if n.endswith("/final_model_selection.json")
        )
        selection = json.loads(eval_zip.read(selection_name))

        actual = sha256_bytes(
            e1_zip.read("best_adapter/adapter_model.safetensors")
        )
        expected = selection["adapter_weights_sha256"]

        print("E1 adapter 与选型记录：", "一致" if actual == expected else "不一致")
        print("选定模型：", selection["selected_experiment"])
        print("选型数据集：", selection["selection_split"])

if data_path and eval_path:
    with zipfile.ZipFile(data_path) as data_zip, \
         zipfile.ZipFile(eval_path) as eval_zip:

        manifest = json.loads(data_zip.read("manifest.json"))
        selection_name = next(
            n for n in eval_zip.namelist()
            if n.endswith("/final_model_selection.json")
        )
        selection = json.loads(eval_zip.read(selection_name))

        actual = sha256_bytes(data_zip.read("val.jsonl"))
        expected = selection["validation_sha256"]

        print("固定 Val 与选型记录：", "一致" if actual == expected else "不一致")
        print("固定 Val 与数据清单：",
              "一致" if actual == manifest["files_sha256"]["val.jsonl"]
              else "不一致")

if test_path:
    # 此哈希来自原 Phase 5 报告，核对的是 ZIP 内的 test.jsonl，
    # 不是整个 ZIP 文件的哈希。
    reported_test_sha256 = (
        "700f21dc709f0eb40f1fcb323f0fa6ee675e7399"
        "c3e9b2fbd77f9f1cf1976473"
    )

    with zipfile.ZipFile(test_path) as test_zip:
        test_name = next(
            n for n in test_zip.namelist()
            if n.endswith("/test.jsonl")
        )
        actual = sha256_bytes(test_zip.read(test_name))

    print("Test 数据与 Phase 5 报告：",
          "一致" if actual == reported_test_sha256 else "不一致")

## 1.2 四方法统一 Validation 结果核验

使用同一份 Validation（1,000 条指令；Task A、B 各 500 条）核对 Rule、Zero-shot、Few-shot 和 E1 LoRA。Zero-shot、Few-shot、LoRA 均有 Phase 4 原始逐题预测；Rule 可由时间比较代码与原题复算。四方法的结果已按 `sample_id` 对齐并重新计数。

| 方法 | Task A 严格 JSON | Task A 严格字段正确 | Task B 正确数 |
|---|---:|---:|---:|
| Rule | 不适用 | 不适用 | 500/500（100%） |
| Zero-shot | 1/500 | 7/4,000（0.18%） | 334/500（66.8%） |
| Few-shot | 500/500 | 3,963/4,000（99.08%） | 311/500（62.2%） |
| E1 LoRA | 500/500 | 4,000/4,000（100%） | 497/500（99.4%） |

**评分口径**：严格指标直接解析原始回答。Zero-shot 常输出 Markdown 代码围栏，因此严格字段仅 7/4,000；只移除完整围栏后，字段正确数为 3,759/4,000（93.97%），该数值属于辅助分析，不替代严格指标。

**文件核对**：Phase 4 LoRA 预测共 1,000 条、编号无重复且与 Val 完全一致；其 SHA-256 为 `02e6e5bbdf487e74903d18f0ba3ba3ad0f0537708cdb9bf650f2e5394369bbe3`，与 Phase 4 Notebook 记录一致。上述结果仅适用于该合成数据验证集。

## 2. 模板分布与分模板表现

**作用**：检查训练规模扩大时，Task B 的模板数量是否失衡。

**方法**：按 template_id 统计 E1/E2/E3 训练指令；再将 E1 Test 的预测按 sample_id 对回原题，统计两种 Test 模板的正确数。

**目的**：补充 E3 错误分析，并确认 E1 Test 是否覆盖了两种时间叙述顺序。

模板编号属于各自的数据集，不能仅凭编号相同就认定文字模板相同。统计结果说明分布和表现，不单独证明错误成因。

In [ ]:
from collections import Counter
import json
import zipfile

def read_jsonl(archive, suffix):
    """按内部文件名后缀读取 JSONL。"""
    matches = [name for name in archive.namelist()
               if name.endswith(suffix)]
    if len(matches) != 1:
        raise ValueError(f"{suffix} 匹配到 {len(matches)} 个文件")
    return [json.loads(line)
            for line in archive.read(matches[0]).splitlines()]

phase4_path = found["FleetMind_LLM_Phase4_Evaluation_Dataset.zip"]
training_path = found["FleetMind_LLM_Phase5_Training_Data.zip"]
evaluation_path = found["FleetMind_LLM_Phase5_Evaluation_Results.zip"]

with zipfile.ZipFile(phase4_path) as phase4_zip, \
     zipfile.ZipFile(training_path) as training_zip, \
     zipfile.ZipFile(evaluation_path) as evaluation_zip:

    training_sets = {
        "E1": read_jsonl(phase4_zip, "/train.jsonl"),
        "E2": read_jsonl(training_zip, "E2_train.jsonl"),
        "E3": read_jsonl(training_zip, "E3_train.jsonl"),
    }

    print("=== 训练集 Task B 模板分布 ===")
    for model, rows in training_sets.items():
        task_b = [row for row in rows if row["task"] == "B"]
        counts = Counter(row["template_id"] for row in task_b)

        print(f"\n{model}：总指令 {len(rows)}，Task B {len(task_b)}")
        for template_id, count in sorted(counts.items()):
            print(f"  {template_id}: {count}/{len(task_b)} "
                  f"({count / len(task_b):.1%})")

    test_rows = read_jsonl(phase4_zip, "/test.jsonl")
    predictions = read_jsonl(
        evaluation_zip, "/predictions/E1_test_predictions.jsonl"
    )

# 用 sample_id 关联原题与预测，而不是依赖文件行顺序
prediction_by_id = {row["sample_id"]: row for row in predictions}
assert len(prediction_by_id) == len(predictions), "预测中有重复 sample_id"

total = Counter()
correct = Counter()

for row in test_rows:
    if row["task"] != "B":
        continue

    prediction = prediction_by_id[row["sample_id"]]
    template_id = row["template_id"]
    total[template_id] += 1

    if json.loads(prediction["prediction"]) == json.loads(
        prediction["reference"]
    ):
        correct[template_id] += 1

print("\n=== E1 Test：Task B 按模板统计 ===")
for template_id in sorted(total):
    print(f"{template_id}: {correct[template_id]}/{total[template_id]} "
          f"({correct[template_id] / total[template_id]:.1%})")

**这一阶段补充了模板分布分析，并核对了 E1、E2、E3 的训练 loss 范围。**

| 核对项 | 结果 |
|---|---|
| 训练模板分布 | E1、E2、E3 的四种 Task B 模板均各占 **25%**；没有发现某一种模板在数量上不足 |
| E1 最终 Test | `test_B_1` 和 `test_B_2` 分别为 **500/500**；两种时间叙述顺序都得到覆盖 |
| E1 训练 loss | 只对**助手答案及结束标记**计算；提示部分的标签设为 `-100` |
| E2、E3 训练 loss | 对**包含提示和答案的完整序列**计算 |

**科学结论**：E3 的错误集中在 `validation_B_2` 是已观察到的现象，但模板数量统计不能解释其成因。E1 与 E2/E3 的 loss 范围不同，因此三者的 loss 数值不能直接比较，也不能把模型表现差异单独归因于训练数据从 1,600 增至 4,000、8,000 条。

Phase 5 发布副本已更正“E2/E3 与 E1 使用相同文本训练口径”的说明。已保存的 Val 和 Test **生成评测结果不因此作废**；需要收紧的是对训练规模实验原因的解释。

## 3. 整理公开仓库

### 3.1 文件范围与目录

| 仓库位置 | 内容 | 来源 |
|---|---|---|
| `notebooks/` | Phase 0–6 的整理版 Notebook | 已保存的七个 Notebook |
| `data/` | E1/E2/E3 训练集、固定 Val、最终 Test、数据说明与哈希 | Phase 4 数据包、Phase 5 训练数据包 |
| `results/phase4/` | Zero-shot、Few-shot、E1 LoRA 的 Val 逐题预测 | Phase 4 原始预测文件 |
| `results/phase5/` | 三模型 Val、E1 Test 的预测、指标与选型记录 | Phase 5 评测结果包 |
| `docs/` | Phase 5 报告与 Phase 6 补充分析 | 已完成的报告和核验结论 |

三个 adapter 和原始结果 ZIP 留作实验档案，发布说明将记录其名称和哈希。公开仓库优先使用解压后的数据、配置和结果，便于逐项查看。

**整理要求**：清理 Notebook 的中断记录和冗长输出；保留实际结果。更正 Phase 5 Notebook 对训练 loss 的说明：E1 只计算答案部分，E2/E3 计算完整序列，不能将三者的 loss 直接比较。

## 6.4 项目首页与证据范围
首页采用统一 Val 对比和选型后的 E1 Test 结果。E1 与 E2/E3 的监督范围不同。E3 的错误按模板报告。困难集题目已按原哈希恢复，逐题预测已归档并复算。